In [1]:
from pathlib import Path

import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

In [3]:
transactions_path = (
    project_root
    / "data"
    / "processed"
    / "transactions_customer_metrics.csv"
)

transactions = pd.read_csv(
    transactions_path,
    parse_dates=[
        "timestamp",
        "opening_date",
        "signup_date",
    ],
)

transactions.shape

(52, 49)

In [4]:
# To create spending profiles I will not use declined transactions and refunds

completed_purchases = transactions.loc[(transactions["status"].eq("Completed")) & (transactions["transaction_type"] == "Purchase")].copy()
completed_purchases.shape

(48, 49)

In [5]:
# Customer category matrix with pivot_table()
# Rows are customers, columns are categories and the values are the total spending

customer_category_matrix = (completed_purchases.pivot_table(
    index = "customer_id",
    columns = "merchant_category",
    values = "absolute_amount",
    aggfunc = "sum",
    fill_value = 0,
)
.sort_index(axis = 1))

customer_category_matrix

merchant_category,Books,Electronics,Entertainment,Fashion,Groceries,Health,Home,Office,Restaurants,Sports,Travel
customer_id,,,,,,,,,,,
C001,19.99,899.00,70.49,76.00,34.80,0.0,0.00,0.00,6.8,0.0,0.0
C002,0.00,423.99,0.00,0.00,0.00,0.0,0.00,12.60,4.2,0.0,815.0
C003,43.98,39.99,0.00,0.00,0.00,0.0,0.00,0.00,0.0,52.0,0.0
C004,0.00,249.90,0.00,0.00,0.00,0.0,58.75,14.25,0.0,82.5,410.0
C005,21.50,0.00,0.00,0.00,62.40,0.0,130.60,0.00,0.0,0.0,0.0
C006,0.00,89.00,0.00,0.00,81.85,0.0,0.00,0.00,0.0,0.0,0.0
C007,0.00,0.00,0.00,88.80,0.00,0.0,0.00,0.00,13.0,0.0,0.0
C008,0.00,0.00,0.00,145.90,71.35,0.0,0.00,0.00,0.0,0.0,945.0
C009,0.00,0.00,59.99,0.00,0.00,0.0,74.50,0.00,0.0,0.0,155.0


In [6]:
matrix_total = (customer_category_matrix.sum().sum())
purchase_total = (completed_purchases["absolute_amount"].sum())

matrix_total == purchase_total

np.True_

In [7]:
customer_category_long = (customer_category_matrix
.reset_index()
.melt(
    id_vars = "customer_id",
    var_name = "merchant_category",
    value_name = "total_amount"
))

In [8]:
customer_category_long

,customer_id,merchant_category,total_amount
0,C001,Books,19.99
1,C002,Books,0.00
2,C003,Books,43.98
3,C004,Books,0.00
4,C005,Books,21.50
...,...,...,...
127,C008,Travel,945.00
128,C009,Travel,155.00
129,C010,Travel,0.00
130,C011,Travel,0.00


In [9]:
# I also eliminate categories with no spending
customer_category_long = (
    customer_category_long.loc[
        customer_category_long[
            "total_amount"
        ].gt(0)
    ]
    .sort_values(
        [
            "customer_id",
            "total_amount",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

customer_category_long.head(15)

,customer_id,merchant_category,total_amount
0,C001,Electronics,899.00
1,C001,Fashion,76.00
2,C001,Entertainment,70.49
3,C001,Groceries,34.80
4,C001,Books,19.99
5,C001,Restaurants,6.80
6,C002,Travel,815.00
7,C002,Electronics,423.99
8,C002,Office,12.60
9,C002,Restaurants,4.20


In [10]:
reconstructed_matrix = (
    customer_category_long
    .pivot(
        index = "customer_id",
        columns = "merchant_category",
        values = "total_amount"
    )
    .fillna(0)
)

In [11]:
reconstructed_matrix.head()

merchant_category,Books,Electronics,Entertainment,Fashion,Groceries,Health,Home,Office,Restaurants,Sports,Travel
customer_id,,,,,,,,,,,
C001,19.99,899.00,70.49,76.0,34.8,0.0,0.00,0.00,6.8,0.0,0.0
C002,0.00,423.99,0.00,0.0,0.0,0.0,0.00,12.60,4.2,0.0,815.0
C003,43.98,39.99,0.00,0.0,0.0,0.0,0.00,0.00,0.0,52.0,0.0
C004,0.00,249.90,0.00,0.0,0.0,0.0,58.75,14.25,0.0,82.5,410.0
C005,21.50,0.00,0.00,0.0,62.4,0.0,130.60,0.00,0.0,0.0,0.0


In [12]:
customer_scope_series = (
    transactions
    .groupby(
        [
            "customer_id",
            "transaction_scope",
        ]
    ) ["net_amount"].sum()
)
customer_scope_series.head()

customer_id  transaction_scope
C001         Domestic               57.10
             International        1029.99
C002         Domestic               16.80
             International        1238.99
C003         International         135.97
Name: net_amount, dtype: float64

In [13]:
customer_scope_series.index.names

FrozenList(['customer_id', 'transaction_scope'])

In [14]:
customer_scope_matrix = (
    customer_scope_series.unstack(fill_value = 0)
)

customer_scope_matrix

transaction_scope,Domestic,International
customer_id,,
C001,57.10,1029.99
C002,16.80,1238.99
C003,0.00,135.97
C004,249.90,565.50
C005,21.50,193.00
C006,0.00,170.85
C007,0.00,101.80
C008,0.00,1162.25
C009,155.00,134.49


In [15]:
customer_scope_long = (
    customer_scope_matrix
    .stack()
    .rename("net_amount")
    .reset_index()
)

customer_scope_long.head()

,customer_id,transaction_scope,net_amount
0,C001,Domestic,57.10
1,C001,International,1029.99
2,C002,Domestic,16.80
3,C002,International,1238.99
4,C003,Domestic,0.00


In [ ]:
# To create spending profiles I will not use declined transactions and refunds

completed_purchases = transactions.loc[(transactions["status"].eq("Completed")) & (transactions["transaction_type"] == "Purchase")].copy()
completed_purchases.shape

(48, 49)

In [22]:
customer_category_count = (
    completed_purchases
    .pivot_table(
        index="customer_id",
        columns="merchant_category",
        values="transaction_id",
        aggfunc="count",
        fill_value=0,
    )
    .sort_index(axis=1)
)

customer_category_count

merchant_category,Books,Electronics,Entertainment,Fashion,Groceries,Health,Home,Office,Restaurants,Sports,Travel
customer_id,,,,,,,,,,,
C001,1,1,2,1,1,0,0,0,1,0,0
C002,0,2,0,0,0,0,0,1,1,0,2
C003,2,1,0,0,0,0,0,0,0,1,0
C004,0,1,0,0,0,0,1,1,0,1,1
C005,1,0,0,0,1,0,1,0,0,0,0
C006,0,1,0,0,2,0,0,0,0,0,0
C007,0,0,0,1,0,0,0,0,2,0,0
C008,0,0,0,1,1,0,0,0,0,0,2
C009,0,0,1,0,0,0,1,0,0,0,1


In [23]:
print(
    customer_category_count
    .sum()
    .sum()
)

print(
    len(completed_purchases)
)

48
48


In [24]:
reshaping_validation = pd.Series(
    {
        "amount_matrix_matches_source": (
            customer_category_matrix
            .sum()
            .sum()
            == completed_purchases[
                "absolute_amount"
            ].sum()
        ),
        "count_matrix_matches_source": (
            customer_category_count
            .sum()
            .sum()
            == len(completed_purchases)
        ),
        "long_table_has_unique_pairs": (
            ~customer_category_long
            .duplicated(
                [
                    "customer_id",
                    "merchant_category",
                ]
            )
            .any()
        ),
        "long_total_matches_matrix": (
            customer_category_long[
                "total_amount"
            ].sum()
            == customer_category_matrix
            .sum()
            .sum()
        ),
        "scope_total_is_preserved": (
            customer_scope_matrix
            .sum()
            .sum()
            == transactions[
                "net_amount"
            ].sum()
        ),
    },
    name="passed",
)

reshaping_validation

amount_matrix_matches_source     True
count_matrix_matches_source      True
long_table_has_unique_pairs      True
long_total_matches_matrix       False
scope_total_is_preserved         True
Name: passed, dtype: bool

In [26]:
# long_total_matches_matrix is False but I check it and it is due to a non significant decimal difference

long_total = customer_category_long[
    "total_amount"
].sum()

matrix_total = (
    customer_category_matrix
    .sum()
    .sum()
)

print("Long total:", repr(long_total))
print("Matrix total:", repr(matrix_total))
print("Difference:", long_total - matrix_total)

Long total: np.float64(5690.219999999999)
Matrix total: np.float64(5690.22)
Difference: -9.094947017729282e-13


In [27]:
customer_category_matrix_path = (
    project_root
    / "data"
    / "processed"
    / "customer_category_matrix.csv"
)

customer_category_long_path = (
    project_root
    / "data"
    / "processed"
    / "customer_category_long.csv"
)

customer_scope_matrix_path = (
    project_root
    / "data"
    / "processed"
    / "customer_scope_matrix.csv"
)

In [28]:
customer_category_matrix.to_csv(
    customer_category_matrix_path
)

customer_category_long.to_csv(
    customer_category_long_path,
    index=False,
)

customer_scope_matrix.to_csv(
    customer_scope_matrix_path
)

## Reshaping results

The transaction data was reorganised into customer level analytical matrices

In this phase I produced:

- A customer * merchant category spending matrix
- A customer * merchant category transaction count matrix
- A long format customer category table
- A customer * transaction scope matrix

The customer category matrix will be converted into a numpy array in the next phase